# Finanzas Públicas Estatales y Municipales (EFIPEM)

En este archivo se lee el monto de inversión pública de 2020 a 2025 por alcaldía de los reportes del INEGI que se pueden consultar en el siguiente enlace.

[Finanzas públicas](https://www.inegi.org.mx/sistemas/olap/consulta/general_ver4/MDXQueryDatos.asp?c=11291)

In [ ]:
# Rutas del proyecto (antes: montar Google Drive). Las define rutas.py, en la
# raíz del repositorio; funciona igual en Docker, en local y en Colab con el repo clonado.
import sys
from pathlib import Path
_raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rutas.py").exists())
sys.path.insert(0, str(_raiz))
from rutas import *

In [ ]:

import pandas as pd
import os
import re
import tempfile
import shutil

CARPETA = str(FINANZAS)

In [ ]:
ARCHIVOS = [
    "INEGI_exporta_3_10_2026_16_41_8_2020.xls",
    "INEGI_exporta_3_10_2026_16_43_54_2021.xls",
    "INEGI_exporta_3_10_2026_17_1_14_2022.xls",
    "INEGI_exporta_3_10_2026_16_46_10_2023.xls",
    "INEGI_exporta_3_10_2026_17_11_9_2024.xls",
    "INEGI_exporta_3_10_2026_16_52_54_2025.xls"
]



In [ ]:
def limpiar_monto(valor):

    if pd.isna(valor):
        return None
    texto = str(valor).strip()
    # Quitar espacios normales y especiales
    texto = texto.replace(" ", "").replace("\xa0", "")
    # Quitar comas
    texto = texto.replace(",", "")

    try:
        return float(texto)
    except:
        return None

In [ ]:
def leer_excel_inegi(ruta):

    archivo_temporal = tempfile.NamedTemporaryFile(
        suffix=".xlsx",
        delete=False
    )

    archivo_temporal.close()

    shutil.copy(ruta, archivo_temporal.name)

    try:

        df = pd.read_excel(
            archivo_temporal.name,
            header=None,
            engine="openpyxl"
        )

    finally:

        os.remove(archivo_temporal.name)

    return df


In [ ]:
# PROCESAR CADA ARCHIVO
def procesar_archivo(ruta):
    nombre = os.path.basename(ruta)
    print(f"\nProcesando: {nombre}")
    # Leer Excel completo sin asumir encabezados
    df_raw = leer_excel_inegi(ruta)
    # Encontrar el año DENTRO del archivo
    año = None

    for valor in df_raw.astype(str).values.flatten():

        match = re.search(
            r"Año\s*:\s*(\d{4})",
            str(valor),
            flags=re.IGNORECASE
        )

        if match:

            año = int(match.group(1))
            break


    print(f"   Año encontrado dentro del archivo: {año}")
    # Encontrar la fila que contiene "Inversión pública"


    fila_inversion = None

    for i in range(len(df_raw)):

        valores_fila = df_raw.iloc[i].astype(str)

        if valores_fila.str.contains(
            "Inversión pública",
            case=False,
            na=False
        ).any():

            fila_inversion = i
            break


    if fila_inversion is None:

        print("   ❌ No encontré la fila 'Inversión pública'")
        return None


    # La fila anterior contiene las alcaldías
    fila_alcaldias = fila_inversion - 1

    alcaldias = df_raw.iloc[fila_alcaldias].tolist()
    valores = df_raw.iloc[fila_inversion].tolist()

    # Construir tabla limpia
    datos = []

    for alcaldia, valor in zip(alcaldias, valores):
        if pd.isna(alcaldia):
            continue
        alcaldia = str(alcaldia).strip()

        if alcaldia == "" or alcaldia.lower() == "nan":
            continue
        monto = limpiar_monto(valor)
        if monto is not None:

            datos.append({

                "año": año,
                "alcaldia": alcaldia,
                "inversion_publica": monto,
                "archivo_origen": nombre

            })


    df_limpio = pd.DataFrame(datos)

    print(f"   ✓ Registros encontrados: {len(df_limpio)}")

    return df_limpio

dfs = []
for archivo in ARCHIVOS:
    # Busca el archivo en finanzasPublicas/ (también en subcarpetas)
    ruta = next((str(p) for p in FINANZAS.rglob(archivo)), None)
    # Si falta un año, la corrida se detiene: un año faltante cambia el panel
    if ruta is None:
        raise FileNotFoundError(f"No encontré {archivo} en {FINANZAS}.")
    try:

        df_temp = procesar_archivo(ruta)

        if df_temp is not None and not df_temp.empty:

            dfs.append(df_temp)

    except Exception as e:

        print(f"❌ Error leyendo {archivo}")
        print(e)




In [ ]:
# 7. UNIR TODO
df_inversion = pd.concat(
    dfs,
    ignore_index=True
)

# Quitar agregado de toda CDMX

df_inversion = df_inversion[
    df_inversion["alcaldia"].str.strip() != "Ciudad de México"
].copy()
#Ordenar
df_inversion = (
    df_inversion
    .sort_values(["año", "alcaldia"])
    .reset_index(drop=True)
)


# Convertir inversión a número entero nullable
df_inversion["inversion_publica"] = (
    df_inversion["inversion_publica"]
    .round()
    .astype("Int64")
)

In [ ]:
print("\n" + "=" * 80)
print("DATAFRAME FINAL")
print("=" * 80)
display(df_inversion)
print("\nAños encontrados:")
print(sorted(df_inversion["año"].dropna().unique()))


print("\nShape:")
print(df_inversion.shape)


print("\nNúmero de alcaldías por año:")

display(
    df_inversion
    .groupby("año")
    .agg(
        alcaldias=("alcaldia", "nunique"),
        observaciones=("alcaldia", "size")
    )
    .reset_index()
)

In [ ]:
# Seleccionar las primeras 4 columnas
df_inversion = df_inversion.iloc[:, :3]

ruta_salida = salida("inversion_publica_alcaldia.csv")

# Guardar
df_inversion.to_csv(ruta_salida, index=False)


print("Archivo guardado: inversion_publica_alcaldia.csv")